In [ ]:
import coffea
import glob
from coffea.nanoevents import NanoEventsFactory, BaseSchema
import awkward as ak
import vector
vector.register_awkward()

import numpy as np
import ROOT
ROOT.gROOT.SetBatch(True)
from ROOT import TFile, TH1F , TH2F ,TH3F, TF1, TCanvas, TMath, TGraph, TLegend
import matplotlib.pyplot as plt
from array import array

# Load events

files = glob.glob(
    "output_condor_20250827_2122/WbWb/had/wzp6_ee_WbWb_had_ecm365/*.root"
)

events = NanoEventsFactory.from_root(
    f"{files[0]}:events",
    schemaclass=BaseSchema,
).events()

for f in files[1:]:
    ev = NanoEventsFactory.from_root(
        f"{f}:events",
        schemaclass=BaseSchema,
    ).events()
    events = ak.concatenate([events, ev])


In [ ]:
def build_vectors_from_events(events):

    # b-quarks
    b_tlv = events.b_tlv
    b_pdg = events.b_PDG

    b_vec = ak.zip({
        "px": b_tlv.fP.fX,
        "py": b_tlv.fP.fY,
        "pz": b_tlv.fP.fZ,
        "E":  b_tlv.fE,
        "pdg_id": b_pdg
    }, with_name="Momentum4D")

    # light quarks
    q_tlv = events.q_tlv
    q_pdg = events.q_PDG

    q_vec = ak.zip({
        "px": q_tlv.fP.fX,
        "py": q_tlv.fP.fY,
        "pz": q_tlv.fP.fZ,
        "E":  q_tlv.fE,
        "pdg_id": q_pdg
    }, with_name="Momentum4D")

    # combine all quarks
    all_quarks = ak.concatenate([b_vec, q_vec], axis=1)

    # jets (jet1_kt_tlv ... jet6_kt_tlv)
    jets = ak.zip({
        "px": ak.concatenate(
            [events[f"jet{i}_kt_tlv"].fP.fX[:, None] for i in range(1, 10)],
            axis=1
        ),
        "py": ak.concatenate(
            [events[f"jet{i}_kt_tlv"].fP.fY[:, None] for i in range(1, 10)],
            axis=1
        ),
        "pz": ak.concatenate(
            [events[f"jet{i}_kt_tlv"].fP.fZ[:, None] for i in range(1, 10)],
            axis=1
        ),
        "E": ak.concatenate(
            [events[f"jet{i}_kt_tlv"].fE[:, None] for i in range(1, 10)],
            axis=1
        ),
    }, with_name="Momentum4D")

    return {
        "b_vec": b_vec,
        "q_vec": q_vec,
        "all_quarks": all_quarks,
        "jets": jets
    }

vectors = build_vectors_from_events(events)

# Access individual components
all_quarks = vectors["all_quarks"]
jets = vectors["jets"]

valid_jets_mask = ~((jets.px == -999) | (jets.py == -999) | (jets.pz == -999) | (jets.E == -999))
valid_jets = ak.mask(jets, valid_jets_mask)
jets = ak.drop_none(valid_jets)

In [ ]:
# --- Step 1: separate quark types using PDG ------------------------
all_q = all_quarks

b_mask      = all_q.pdg_id == 5
bbar_mask   = all_q.pdg_id == -5
light_mask  = np.abs(all_q.pdg_id) < 5   # u,d,s and anti

b      = all_q[b_mask]
bbar   = all_q[bbar_mask]
light  = all_q[light_mask]

valid_event = (ak.num(b)==1) & (ak.num(bbar)==1) & (ak.num(light)==4)

b     = b[valid_event]
bbar  = bbar[valid_event]
light = light[valid_event]


q_top   = light[:,0:2]
print("q_top",q_top.pdg_id)
q_atop  = light[:,2:4]
print("q_atop",q_atop.pdg_id)

# --- Step 3: reconstruct top masses ------------------------------

top_tlv  = b + q_top[:,0] + q_top[:,1]
atop_tlv = bbar + q_atop[:,0] + q_atop[:,1]

m_top  = top_tlv.mass
m_atop = atop_tlv.mass

list_gen_M = list(m_top) + list(m_atop)

# --- flatten mass array ---
masses = ak.flatten(list_gen_M)
masses_np = np.array(masses, dtype=np.float64)

# --- Create ROOT file ---
root_file = ROOT.TFile("gen_top_mass.root", "RECREATE")

# --- Create histogram ---
nbins = 200
xmin  = 100
xmax  = 200

hist = ROOT.TH1F("h_gen_top_mass",
                 "Gen-level reconstructed top mass;Mass [GeV];Entries",
                 nbins, xmin, xmax)

# --- Fill histogram ---
for m in masses_np:
    hist.Fill(m)

# --- Write and close ---
hist.Write()
root_file.Close()

print("ROOT file saved as gen_top_mass.root")
import ROOT

f = ROOT.TFile("gen_top_mass.root")
h = f.Get("h_gen_top_mass")

max_bin = h.GetMaximumBin()
peak_mass_root = h.GetBinCenter(max_bin)
print("Peak from ROOT histogram:", peak_mass_root)


In [ ]:
def match_jets_to_quarks(jets, all_quarks, dR_threshold=0.3):
    """
    Matches jets to quarks based on minimum ΔR distance.

    Parameters:
        jets (ak.Array): Array of jets (with Momentum4D behavior).
        all_quarks (ak.Array): Array of quarks (with Momentum4D behavior).
        dR_threshold (float): Maximum ΔR for a jet to be matched to a quark.

    Returns:
        matched_pairs (ak.Array): Array of matched pairs with structure:
            [{"jet": ..., "quark": ...}, ...]
    """
    # Cartesian product of jets and quarks per event
    jet_quark_pairs = ak.cartesian([jets, all_quarks], axis=1, nested=True)
    jets_for_dR, quarks_for_dR = ak.unzip(jet_quark_pairs)

    # Calculate deltaR between each jet and quark
    jet_quark_dR = jets_for_dR.deltaR(quarks_for_dR)

    # Find minimum deltaR and its index for each jet
    min_dRs = ak.min(jet_quark_dR, axis=2)
    min_indices = ak.argmin(jet_quark_dR, axis=2)

    # Mask jets that are matched to a quark within the threshold
    match_mask = min_dRs < dR_threshold
    matched_indices = min_indices[match_mask]
    matched_quarks = all_quarks[matched_indices]
    matched_jets = jets[match_mask]

    # Create zip of matched pairs
    matched_pairs = ak.zip({"jet": matched_jets, "quark": matched_quarks})

    return matched_pairs

# valid_quarks = all_quarks[mask_ttbar]
# Step 1: Match jets to quarks using deltaR matching with a threshold of 0.3
matched_pairs = match_jets_to_quarks(jets, all_quarks , dR_threshold=0.3)

# Step 2: Zip matched jets with their corresponding quark PDG IDs
jets_with_pdg = ak.zip({
    "jet": matched_pairs.jet,      # Jet 4-vectors
    "pdg_id": matched_pairs.quark.pdg_id      # PDG ID of matched quark
})

# Step 3: Form all combinations of 3 jets per event (triplets)
triplets = ak.combinations(jets_with_pdg, 3)

# Step 4: Extract PDG IDs from the 3 jets in each triplet
triplet_pdgs = ak.zip({
    "a": triplets["0"].pdg_id,
    "b": triplets["1"].pdg_id,
    "c": triplets["2"].pdg_id
})

def contains_all_elements(triplet_pdgs, required):
    # Check if each required value is present in any position of the triplet
    return ak.all([
        (triplet_pdgs.a == val) | (triplet_pdgs.b == val) | (triplet_pdgs.c == val)
        for val in required
    ], axis=0)

# Step 6: Define top quark triplet signatures (b + light quark + anti-quark)
top_mask1 = contains_all_elements(triplet_pdgs, [5, 4, -3])   
top_mask2 = contains_all_elements(triplet_pdgs, [5, 2, -3])   
top_mask3 = contains_all_elements(triplet_pdgs, [5, 4, -1])   
top_mask4 = contains_all_elements(triplet_pdgs, [5, 2, -1])   

# Step 7: Combine all top masks
top_mask = top_mask1 | top_mask2 | top_mask3 | top_mask4

# Step 8: Apply mask to select top-like jet triplets
top_triplets = triplets[top_mask]

# Step 9: Define anti-top quark triplet signatures (b̄ + light anti-quark + quark)
atop_mask1 = contains_all_elements(triplet_pdgs, [-5, -4, 3]) 
atop_mask2 = contains_all_elements(triplet_pdgs, [-5, -2, 3]) 
atop_mask3 = contains_all_elements(triplet_pdgs, [-5, -4, 1]) 
atop_mask4 = contains_all_elements(triplet_pdgs, [-5, -2, 1]) 

# Step 10: Combine all anti-top masks
atop_mask = atop_mask1 | atop_mask2 | atop_mask3 | atop_mask4

# Step 11: Apply mask to select anti-top-like jet triplets
atop_triplets = triplets[atop_mask]

# Extract top and anti-top masses
top_p4 = top_triplets["0"].jet + top_triplets["1"].jet + top_triplets["2"].jet
atop_p4 = atop_triplets["0"].jet + atop_triplets["1"].jet + atop_triplets["2"].jet

top_mass = top_p4.mass
atop_mass = atop_p4.mass

# Require events to have exactly one top and one anti-top triplet
mask_ttbar = (ak.num(top_triplets) == 1) & (ak.num(atop_triplets) == 1)

# Apply the mask to select valid top and anti-top triplets
top_triplets_valid = top_triplets[mask_ttbar]
atop_triplets_valid = atop_triplets[mask_ttbar]

# Reconstruct the 4-vectors of top and anti-top by summing the jets in each triplet
top_p4_valid = top_triplets_valid["0"].jet + top_triplets_valid["1"].jet + top_triplets_valid["2"].jet
atop_p4_valid = atop_triplets_valid["0"].jet + atop_triplets_valid["1"].jet + atop_triplets_valid["2"].jet

# Compute ttbar system 4-vector
ttbar_p4 = top_p4_valid + atop_p4_valid

# Extract ttbar system mass
ttbar_mass = ttbar_p4.mass

#_____________________
top_mass_valid = top_p4_valid.mass
atop_mass_valid = atop_p4_valid.mass

# Get indices of jet triplets selected as top and anti-top candidates
indices_triplets = ak.argcombinations(jets_with_pdg, 3)
top_indices = indices_triplets[top_mask]
atop_indices = indices_triplets[atop_mask]

def build_W_with_pdgs(triplets, b_pdg_id):
    # Extract PDG IDs and jet 4-vectors from each element in the triplet
    pdg0 = triplets["0"].pdg_id
    pdg1 = triplets["1"].pdg_id
    pdg2 = triplets["2"].pdg_id

    jet0 = triplets["0"].jet
    jet1 = triplets["1"].jet
    jet2 = triplets["2"].jet

    # Create boolean masks to identify which jet in the triplet is the b-jet
    is_b_0 = pdg0 == b_pdg_id
    is_b_1 = pdg1 == b_pdg_id
    is_b_2 = pdg2 == b_pdg_id

    # Build the W boson 4-vector by summing the two non-b jets in the triplet
    W_vec = ak.where(
        is_b_0, jet1 + jet2,
        ak.where(is_b_1, jet0 + jet2, jet0 + jet1)
    )

    W_pdgs = ak.zip({
        "a": ak.where(is_b_0, pdg1, ak.where(is_b_1, pdg0, pdg0)),
        "b": ak.where(is_b_0, pdg2, ak.where(is_b_1, pdg2, pdg1))
    })

    return W_vec, W_pdgs

# Build W bosons from top and anti-top triplets using the b-quark PDG IDs
W_top_vec, W_top_pdgs = build_W_with_pdgs(top_triplets, b_pdg_id=5)
W_atop_vec, W_atop_pdgs = build_W_with_pdgs(atop_triplets, b_pdg_id=-5)

# Build W bosons only for valid top/anti-top combinations (e.g. selected via PDF)
W_top_vec_valid, W_top_pdgs_valid = build_W_with_pdgs(top_triplets_valid, b_pdg_id=5)
W_atop_vec_valid, W_atop_pdgs_valid = build_W_with_pdgs(atop_triplets_valid, b_pdg_id=-5)

# Extract W boson masses from the 4-vectors
W_top_mass = W_top_vec.mass
W_atop_mass = W_atop_vec.mass

W_top_mass_valid = W_top_vec_valid.mass
W_atop_mass_valid = W_atop_vec_valid.mass


In [ ]:
# ============================================================
# Efficiency / Event Selection Summary
# ============================================================

# ------------------------------------------------------------
# 1. Total number of valid jets
# ------------------------------------------------------------

n_initial_events = len(events)

# Number of valid jets in all events
n_total_jets = ak.sum(ak.num(jets))

mean_jets_per_event = n_total_jets / n_initial_events

print("=" * 60)
print("1. JET INFORMATION")
print("=" * 60)
print(f"Total number of events          : {n_initial_events}")
print(f"Total number of valid jets      : {n_total_jets}")
print(f"Mean number of jets per event   : {mean_jets_per_event:.3f}")


# ------------------------------------------------------------
# 2. Events with at least one triplet
# ------------------------------------------------------------

has_triplet = ak.num(triplets) > 0

n_events_with_triplet = ak.sum(has_triplet)

eff_triplet = n_events_with_triplet / n_initial_events

print("\n" + "=" * 60)
print("2. TRIPLET SELECTION")
print("=" * 60)
print(f"Events with >= 1 triplet        : {n_events_with_triplet}")
print(f"Efficiency                      : {eff_triplet:.6f}")
print(f"Efficiency (%)                  : {eff_triplet * 100:.3f}%")


# ------------------------------------------------------------
# 3. Events with at least one top OR anti-top candidate
# ------------------------------------------------------------

has_top = ak.num(top_triplets) > 0
has_atop = ak.num(atop_triplets) > 0

has_at_least_one_top = has_top | has_atop

n_events_at_least_one_top = ak.sum(has_at_least_one_top)

# Efficiency relative to events with at least one triplet
eff_one_top = (
    n_events_at_least_one_top / n_events_with_triplet
)

print("\n" + "=" * 60)
print("3. TOP / ANTI-TOP SELECTION")
print("=" * 60)
print(f"Events with >= 1 top/anti-top   : {n_events_at_least_one_top}")
print(f"Efficiency                      : {eff_one_top:.6f}")
print(f"Efficiency (%)                  : {eff_one_top * 100:.3f}%")


# ------------------------------------------------------------
# 4. Events with exactly one top AND one anti-top
# ------------------------------------------------------------

mask_ttbar = (
    (ak.num(top_triplets) == 1)
    &
    (ak.num(atop_triplets) == 1)
)

n_events_ttbar = ak.sum(mask_ttbar)

# Efficiency relative to events with at least one top/anti-top
eff_ttbar = (
    n_events_ttbar / n_events_at_least_one_top
)

print("\n" + "=" * 60)
print("4. t tbar SELECTION")
print("=" * 60)
print(f"Events with exactly 1 top + 1 anti-top : {n_events_ttbar}")
print(f"Efficiency                             : {eff_ttbar:.6f}")
print(f"Efficiency (%)                         : {eff_ttbar * 100:.3f}%")


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)

print(f"Initial events                         : {n_initial_events}")
print(f"Events with >= 1 triplet               : {n_events_with_triplet}")
print(f"Events with >= 1 top/anti-top          : {n_events_at_least_one_top}")
print(f"Events with exactly 1 top + 1 anti-top : {n_events_ttbar}")

print("\nEfficiencies:")
print(f"Triplet / Initial                      : {eff_triplet * 100:.3f}%")
print(f">=1 top / Triplet                      : {eff_one_top * 100:.3f}%")
print(f"1 top + 1 anti-top / >=1 top           : {eff_ttbar * 100:.3f}%")

# Overall efficiency from initial sample to exactly one top + one anti-top
overall_eff_ttbar = n_events_ttbar / n_initial_events

print(f"Overall: t tbar / Initial              : {overall_eff_ttbar * 100:.3f}%")

In [ ]:

f = TFile("top_profile_for_paper.root", "RECREATE")

hist_top = TH1F("hist_top", "Top-AntiTop Masses;Mass [GeV];Events", 50, 50, 220)
deltaM_top = TH1F("deltaM_top", "Top-AntiTop Mass difference;deltaMass [GeV];Events", 50, -100, 100)

hist_W = TH1F("W_mass", "W and AntiW Mass;Mass [GeV];Entries", 80,40,120)
deltaM_W = TH1F("deltaM_W", "W-AntiW Mass difference;deltaMass [GeV];Events", 50, -50, 50)

ttbar = TH1F("M_ttbar", "M_ttbar;Mass_ttbar [GeV];Entries", 40, 220, 380)

# jet5_p = TH1F("jet5_p", "jet5_p ;p[GeV];Entries", 150, 0, 150)
# jet6_p = TH1F("jet6_p", "jet6_p ;p[GeV];Entries", 150, 0, 150)


top_vs_ttbar = TH2F("top_vs_ttbar", "Top Mass vs ttbar Mass;ttbar Mass [GeV];Top and AntiTop Masses [GeV]", 
                40, 220, 380,   # X axis: ttbar mass
                40, 100, 300)   # Y axis: Top mass

top_vs_atop = TH2F("top_vs_antitop", "Top Mass vs AntiTop Mass;AntiTop Mass [GeV];Top Mass [GeV]", 
                40, 100, 300,   # X axis: aTop mass
                40, 100, 300)   # Y axis: Top mass

W_vs_aW = TH2F("W_vs_aW", "W Mass vs AntiW Mass;AntiW Mass [GeV];W Mass [GeV]", 
                80, 40, 120,   # X axis: aW mass
                80, 40, 120)   # Y axis: W mass


top_vs_W = TH2F("top_vs_W", "Top Mass vs W Mass;W Mass [GeV];Top Mass [GeV]", 
                80, 40, 120,   # X axis: W mass
                40, 100, 300)  # Y axis: Top mass

hist3D = TH3F("hist3D", "3D Histogram;M_ttbar [GeV];M_top [GeV];M_W [GeV]", 
              40, 220, 380,  #  x: ttbar
              50, 100, 300,  #  y: top 
              80, 40, 120,   #  z: w
             )     


flat_ttbar = ak.flatten(ttbar_mass)
flat_top   = ak.flatten(top_mass_valid)
flat_atop  = ak.flatten(atop_mass_valid)
flat_W     = ak.flatten(W_top_mass_valid)
flat_aW    = ak.flatten(W_atop_mass_valid)



for tt, mt , mat , mw ,maw in zip(flat_ttbar, flat_top, flat_atop, flat_W, flat_aW ):
    hist3D.Fill(tt, mt, mw)
    hist3D.Fill(tt, mat, maw)

for mass in flat_top :
    hist_top.Fill(mass)    
for mass in flat_atop:
    hist_top.Fill(mass)
    
for dmass in (flat_top - flat_atop):
    deltaM_top.Fill(dmass)

for m in flat_W :
    hist_W.Fill(m)
for m in flat_aW :
    hist_W.Fill(m)

for dmass in (flat_W - flat_aW ):
    deltaM_W.Fill(dmass)
    
for m in flat_ttbar:
    ttbar.Fill(m)
    
# for p in selected_quarks[:,5].p:
#     jet6_p.Fill(p)
# for p in selected_quarks[:,4].p:
#     jet5_p.Fill(p)
    
for tt, top_mass, atop_mass in zip(flat_ttbar, flat_top, flat_atop):
    top_vs_ttbar.Fill(tt, top_mass)   # Fill for top
    top_vs_ttbar.Fill(tt, atop_mass)  # Fill for antitop

for w, t, in zip(flat_W , flat_top):
    top_vs_W.Fill(w, t)

for at, t, in zip(flat_top, flat_atop):
    top_vs_atop.Fill(at, t)
    
for aw, w, in zip(flat_W , flat_aW ):
    W_vs_aW.Fill(aw, w)
    
hist3D.Write()       
hist_top.Write()
deltaM_top.Write()

hist_W.Write()
deltaM_W.Write()

ttbar.Write()

# jet5_p.Write()
# jet6_p.Write()

top_vs_ttbar.Write()
top_vs_atop.Write()
top_vs_W.Write() 
W_vs_aW.Write()

f.Close()

In [ ]:
matched_pairs_valid = matched_pairs
j = matched_pairs_valid.jet.E
q = matched_pairs_valid.quark.E

diff = (j-q) / j


N = len(diff)
mean_val = np.mean(diff)
std_val  = np.std(diff)


fa = TFile("correction2.root", "RECREATE")

xedges = array('d', [0,5,10,11,12,13,14,15,16,17,18,19,20,20.5,21,21.5,22,22.5,23,23.5,24,24.5,25,25.5,26,26.5,27,27.5,28,28.5,29,29.5,30,30.5,31,31.5,32,32.5,33,33.5,34,34.5,35,
                     35.5,36,36.5,37,37.5,38,38.5,39,39.5,40,40.5,41,41.5,42,42.5,43,43.5,44,44.5,45,45.5,46,46.5,47,47.5,48,48.5,49,49.5,50,50.5,
                     51,51.5,52,52.5,53,53.5,54,54.5,55,55.5,56,56.5,57,57.5,58,58.5,59,59.5,60,60.5,61,
                     61.5,62,62.5,63,63.5,64,64.5,65,65.5,66,66.5,67,67.5,68,68.5,69,69.5,70,70.5,71,71.5,72,72.5,73,73.5,
                     74,74.5,75,75.5,76,76.5,77,77.5,78,78.5,79,79.5,80,80.5,81,81.5,82,82.5,83,83.5,84,84.5,85,
                     85.5,86,86.5,87,87.5,88,88.5,89,89.5,90,90.5,91,91.5,92,92.5,93,93.5,94,94.5,95,95.5,96,96.5,97,97.5,
                     98,98.5,99,99.5,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,
                     121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,150,151,152,153,154,155,156,157,158,159,160,
                     165,170,175,180,185])   
# xedges = np.concatenate([
#     np.arange(0, 21, 0.5),
#     np.arange(20, 90.001, 0.25),
#     np.arange(90, 186, 0.5)
# ])
# xedges = array('d', xedges)
              
yedges = array('d', [-10.5,-10,-8,-6,-4,-2,-1,-0.9,-0.8,-0.7,-0.6,-0.5,-0.4,-0.3,-0.2,-0.1,-0.09,-0.08,-0.07,-0.06,-0.05,-0.04,-0.03,-0.02,-0.01,0,
                     0.01,0.02,0.03,0.04,0.05,0.06,0.07,0.08,0.09,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1,1.5,2])    


nx = len(xedges) - 1
ny = len(yedges) - 1

# Energy_correction = TH2F("Energy_correction", "Energy_correction;jet_E [GeV];j_E - q_E / j_E[GeV]", 
#                 nx, xedges,   # X axis: j_E
#                 ny, yedges)   # Y axis: j_E - q_E / q_j

Energy_correction = TH2F(
    "Energy_correction",
    "Jet Energy Response;Reconstructed Jet Energy E_{jet} [GeV];(E_{jet} - E_{quark}) / E_{jet}",
    nx, xedges,
    ny, yedges
)

for ji , d in zip(ak.flatten(j) , ak.flatten(diff)):
    Energy_correction.Fill(ji, d)

Energy_correction.Write()
fa.Close()

In [ ]:
f = ROOT.TFile("correction2.root")
h2 = f.Get("Energy_correction")

#  TProfile
prof = h2.ProfileX("EnergyCorrection_TProfile")

#c = ROOT.TCanvas("c", "Energy Correction Profile", 800, 600)

prof.SetLineColor(ROOT.kRed)
prof.SetMarkerStyle(20)
prof.SetMarkerColor(ROOT.kBlue)
#prof.Draw()

#c.Draw()
fout= ROOT.TFile("out.root", "recreate")
fout.cd()
prof.Write()
fout.Close()

#c.SaveAs("tp.png")

%jsroot on

In [ ]:
# nbins = prof.GetNbinsX()
# edges = np.array([prof.GetBinLowEdge(i) for i in range(1, nbins+2)])
# values = np.array([prof.GetBinContent(i) for i in range(1, nbins+1)])

# def correction_vectorized(E):
#     E_flat = ak.to_numpy(ak.flatten(E))
#     bins = np.digitize(E_flat, edges) - 1
#     bins = np.clip(bins, 0, nbins-1)
#     return values[bins]

valid_jets = jets
# delta_flat = correction_vectorized(valid_jets.E)
# delta = ak.unflatten(delta_flat, ak.num(valid_jets))
# scale = 1 - delta

delta2 = []
for x in ak.flatten(valid_jets.E) :
    if x<25: #<20:
        delta = -0.007235*(x)**2 + 0.341*(x) -4.346
        #delta = 0.1*(x) -2.5
        delta2.append(1-delta)
    if 25 <= x <= 120: #20<=x<=115
        delta = 0.0000015*(x)*(x)*(x) -0.0003*(x)**2 + 0.024*(x) - 0.75
        # if 115 <= x < 119:
        #     delta = 0
        delta2.append(1-delta)


    if 120 < x :
        delta = 0.4
        delta2.append(1-delta)

delta = ak.unflatten(delta2, ak.num(valid_jets))  
#scale = 1 - delta     
px_new = valid_jets.px * delta
py_new = valid_jets.py * delta
pz_new = valid_jets.pz * delta
E_new  = valid_jets.E  * delta

# Build corrected 4-vector array
corrected_jets = ak.zip(
        {"px": px_new, "py": py_new, "pz": pz_new, "E": E_new},
        with_name="Momentum4D")

p_mask = corrected_jets.p > 12
jets_p12 = corrected_jets[p_mask]
has_6_good_jets = ak.num(jets_p12) >= 6
final_event_mask = has_6_good_jets


new_corrected_jets = corrected_jets[final_event_mask]

jets_Energy = new_corrected_jets.E

j1_e = jets_Energy[:,0]
j2_e = jets_Energy[:,1]
j3_e = jets_Energy[:,2]
j4_e = jets_Energy[:,3]
j5_e = jets_Energy[:,4]
j6_e = jets_Energy[:,5]


plt.figure(figsize=(8,6))

plt.hist(j1_e, histtype='step', bins=10, color="red",    label="Jet 1")
plt.hist(j2_e, histtype='step', bins=10, color="blue",   label="Jet 2")
plt.hist(j3_e, histtype='step', bins=10, color="gold",   label="Jet 3")
plt.hist(j4_e, histtype='step', bins=10, color="green",  label="Jet 4")
plt.hist(j5_e, histtype='step', bins=10, color="orange", label="Jet 5")
plt.hist(j6_e, histtype='step', bins=10, color="pink",   label="Jet 6")

plt.xlabel("Jet Energy [GeV]")
plt.ylabel("Entries")
# plt.title("Jet Momentum Distribution After Energy Correction")

plt.legend()
plt.grid(alpha=0.3)

plt.show()


In [ ]:
valid_quarks = all_quarks
# Step 1: Match jets to quarks using deltaR matching with a threshold of 0.3
matched_pairs_corrected = match_jets_to_quarks(corrected_jets, valid_quarks , dR_threshold=0.3)

# Step 2: Zip matched jets with their corresponding quark PDG IDs
jets_with_pdg_corrected = ak.zip({
    "jet": matched_pairs_corrected.jet,      # Jet 4-vectors
    "pdg_id": matched_pairs_corrected.quark.pdg_id      # PDG ID of matched quark
})

# Step 3: Form all combinations of 3 jets per event (triplets)
triplets_corrected = ak.combinations(jets_with_pdg_corrected, 3)

# Step 4: Extract PDG IDs from the 3 jets in each triplet
triplet_pdgs_corrected = ak.zip({
    "a": triplets_corrected["0"].pdg_id,
    "b": triplets_corrected["1"].pdg_id,
    "c": triplets_corrected["2"].pdg_id
})


# Step 6: Define top quark triplet signatures (b + light quark + anti-quark)
top_mask1_corrected = contains_all_elements(triplet_pdgs_corrected, [5, 4, -3])   
top_mask2_corrected = contains_all_elements(triplet_pdgs_corrected, [5, 2, -3])   
top_mask3_corrected = contains_all_elements(triplet_pdgs_corrected, [5, 4, -1])   
top_mask4_corrected = contains_all_elements(triplet_pdgs_corrected, [5, 2, -1])   

# Step 7: Combine all top masks
top_mask_corrected = top_mask1_corrected | top_mask2_corrected | top_mask3_corrected | top_mask4_corrected

# Step 8: Apply mask to select top-like jet triplets
top_triplets_corrected = triplets_corrected[top_mask_corrected]

# Step 9: Define anti-top quark triplet signatures (b̄ + light anti-quark + quark)
atop_mask1_corrected = contains_all_elements(triplet_pdgs_corrected, [-5, -4, 3]) 
atop_mask2_corrected = contains_all_elements(triplet_pdgs_corrected, [-5, -2, 3]) 
atop_mask3_corrected = contains_all_elements(triplet_pdgs_corrected, [-5, -4, 1]) 
atop_mask4_corrected = contains_all_elements(triplet_pdgs_corrected, [-5, -2, 1]) 

# Step 10: Combine all anti-top masks
atop_mask_corrected = atop_mask1_corrected | atop_mask2_corrected | atop_mask3_corrected | atop_mask4_corrected

# Step 11: Apply mask to select anti-top-like jet triplets
atop_triplets_corrected = triplets_corrected[atop_mask_corrected]

# Extract top and anti-top masses
top_p4_corrected = top_triplets_corrected["0"].jet + top_triplets_corrected["1"].jet + top_triplets_corrected["2"].jet
atop_p4_corrected = atop_triplets_corrected["0"].jet + atop_triplets_corrected["1"].jet + atop_triplets_corrected["2"].jet

top_mass_corrected = top_p4_corrected.mass
atop_mass_corrected = atop_p4_corrected.mass

# Require events to have exactly one top and one anti-top triplet
mask_ttbar_corrected = (ak.num(top_triplets_corrected) == 1) & (ak.num(atop_triplets_corrected) == 1)

# Apply the mask to select valid top and anti-top triplets
top_triplets_valid_corrected = top_triplets_corrected[mask_ttbar_corrected]
atop_triplets_valid_corrected = atop_triplets_corrected[mask_ttbar_corrected]

# Reconstruct the 4-vectors of top and anti-top by summing the jets in each triplet
top_p4_valid_corrected = top_triplets_valid_corrected["0"].jet + top_triplets_valid_corrected["1"].jet + top_triplets_valid_corrected["2"].jet
atop_p4_valid_corrected = atop_triplets_valid_corrected["0"].jet + atop_triplets_valid_corrected["1"].jet + atop_triplets_valid_corrected["2"].jet

# Compute ttbar system 4-vector
ttbar_p4_corrected = top_p4_valid_corrected + atop_p4_valid_corrected

# Extract ttbar system mass
ttbar_mass_corrected = ttbar_p4_corrected.mass

#_____________________
top_mass_valid_corrected = top_p4_valid_corrected.mass
atop_mass_valid_corrected = atop_p4_valid_corrected.mass

# Get indices of jet triplets selected as top and anti-top candidates
indices_triplets_corrected = ak.argcombinations(jets_with_pdg_corrected, 3)
top_indices_corrected = indices_triplets_corrected[top_mask_corrected]
atop_indices_corrected = indices_triplets_corrected[atop_mask_corrected]


# Build W bosons from top and anti-top triplets using the b-quark PDG IDs
W_top_vec_corrected, W_top_pdgs_corrected = build_W_with_pdgs(top_triplets_corrected, b_pdg_id=5)
W_atop_vec_corrected, W_atop_pdgs_corrected = build_W_with_pdgs(atop_triplets_corrected, b_pdg_id=-5)

# Build W bosons only for valid top/anti-top combinations (e.g. selected via PDF)
W_top_vec_valid_corrected, W_top_pdgs_valid_corrected = build_W_with_pdgs(top_triplets_valid_corrected, b_pdg_id=5)
W_atop_vec_valid_corrected, W_atop_pdgs_valid_corrected = build_W_with_pdgs(atop_triplets_valid_corrected, b_pdg_id=-5)

# Extract W boson masses from the 4-vectors
W_top_mass_corrected = W_top_vec_corrected.mass
W_atop_mass_corrected = W_atop_vec_corrected.mass

W_top_mass_valid_corrected = W_top_vec_valid_corrected.mass
W_atop_mass_valid_corrected = W_atop_vec_valid_corrected.mass


In [ ]:
# Create a new ROOT file
f_corr = TFile("top_profile_corrected.root", "RECREATE")

# 1D Histograms
hist_top_corr = TH1F("hist_top_corr", "Top-AntiTop Masses;Mass [GeV];Events", 50, 50, 220)
deltaM_top_corr = TH1F("deltaM_top_corr", "Top-AntiTop Mass difference;deltaMass [GeV];Events", 50, -100, 100)

hist_W_corr = TH1F("W_mass_corr", "W and AntiW Mass;Mass [GeV];Entries", 80, 40, 120)
deltaM_W_corr = TH1F("deltaM_W_corr", "W-AntiW Mass difference;deltaMass [GeV];Events", 50, -50, 50)

ttbar_corr = TH1F("M_ttbar_corr", "M_ttbar;Mass_ttbar [GeV];Entries", 40, 220, 400)

# 2D Histograms
top_vs_ttbar_corr = TH2F("top_vs_ttbar_corr", "Top Mass vs ttbar Mass;ttbar Mass [GeV];Top and AntiTop Masses [GeV]", 
                         40, 220, 400, 40, 100, 300)

top_vs_atop_corr = TH2F("top_vs_antitop_corr", "Top Mass vs AntiTop Mass;AntiTop Mass [GeV];Top Mass [GeV]", 
                        40, 100, 300, 40, 100, 300)

W_vs_aW_corr = TH2F("W_vs_aW_corr", "W Mass vs AntiW Mass;AntiW Mass [GeV];W Mass [GeV]", 
                    80, 40, 120, 80, 40, 120)

top_vs_W_corr = TH2F("top_vs_W_corr", "Top Mass vs W Mass;W Mass [GeV];Top Mass [GeV]", 
                     80, 40, 120, 40, 100, 300)

# 3D Histogram
hist3D_corr = TH3F("hist3D_corr", "3D Histogram;M_ttbar [GeV];M_top [GeV];M_W [GeV]", 
                   40, 220, 380, 50, 100, 300, 80, 40, 120)

# Flatten arrays for easy iteration
flat_ttbar_corr = ak.flatten(ttbar_mass_corrected)
flat_top_corr   = ak.flatten(top_mass_valid_corrected)
flat_atop_corr  = ak.flatten(atop_mass_valid_corrected)
flat_W_corr     = ak.flatten(W_top_mass_valid_corrected)
flat_aW_corr    = ak.flatten(W_atop_mass_valid_corrected)

# Fill 3D histogram
for tt, mt, mat, mw, maw in zip(flat_ttbar_corr, flat_top_corr, flat_atop_corr, flat_W_corr, flat_aW_corr):
    hist3D_corr.Fill(tt, mt, mw)
    hist3D_corr.Fill(tt, mat, maw)

# Fill 1D histograms
for mass in flat_top_corr:
    hist_top_corr.Fill(mass)
for mass in flat_atop_corr:
    hist_top_corr.Fill(mass)

for dmass in (flat_top_corr - flat_atop_corr):
    deltaM_top_corr.Fill(dmass)

for m in flat_W_corr:
    hist_W_corr.Fill(m)
for m in flat_aW_corr:
    hist_W_corr.Fill(m)

for dmass in (flat_W_corr - flat_aW_corr):
    deltaM_W_corr.Fill(dmass)

for m in flat_ttbar_corr:
    ttbar_corr.Fill(m)

# Fill 2D histograms
for tt, top_mass, atop_mass in zip(flat_ttbar_corr, flat_top_corr, flat_atop_corr):
    top_vs_ttbar_corr.Fill(tt, top_mass)
    top_vs_ttbar_corr.Fill(tt, atop_mass)

for w, t in zip(flat_W_corr, flat_top_corr):
    top_vs_W_corr.Fill(w, t)

for at, t in zip(flat_top_corr, flat_atop_corr):
    top_vs_atop_corr.Fill(at, t)

for aw, w in zip(flat_W_corr, flat_aW_corr):
    W_vs_aW_corr.Fill(aw, w)

# Write histograms to file
hist3D_corr.Write()
hist_top_corr.Write()
deltaM_top_corr.Write()
hist_W_corr.Write()
deltaM_W_corr.Write()
ttbar_corr.Write()
top_vs_ttbar_corr.Write()
top_vs_atop_corr.Write()
top_vs_W_corr.Write()
W_vs_aW_corr.Write()

f_corr.Close()

In [ ]:
%jsroot off

hist3D = ROOT.TH3F("hist3D_variable_binning", "3D Histogram;delta_M [GeV];M_{top} [GeV];M_{W} [GeV]",
    200, -100, 100,
    200,  80,  230,
    200,  30,  140 
)

for tt, mt , mat , mw , maw in zip(
        ak.flatten(top_mass_valid_corrected-atop_mass_valid_corrected),
        ak.flatten(top_mass_valid_corrected),
        ak.flatten(atop_mass_valid_corrected),
        ak.flatten(W_top_mass_valid_corrected),
        ak.flatten(W_atop_mass_valid_corrected)):

    hist3D.Fill(tt, mt, mw)
    hist3D.Fill(tt, mat, maw)

total = hist3D.Integral("width")
hist3D.Scale(1.0 / total)




f = ROOT.TFile("PDF_dmt_top_w.root", "RECREATE")
hist3D.Write()  


proj_x_y = hist3D.Project3D("yx")  # ΔM vs M_top
proj_y_z = hist3D.Project3D("zy")  # M_top vs M_W
proj_x_z = hist3D.Project3D("zx")  # ΔM vs M_W


proj_x_y.SetTitle("ΔM vs M_top; ΔM [GeV]; M_top [GeV]")
proj_y_z.SetTitle("M_top vs M_W; M_top [GeV]; M_W [GeV]")
proj_x_z.SetTitle("ΔM vs M_W; ΔM [GeV]; M_W [GeV]")


proj_x_y.Write("proj_x_y")
proj_y_z.Write("proj_y_z")
proj_x_z.Write("proj_x_z")

f.Close()

print("Integral with width :", hist3D.Integral("width"))

In [ ]:
selected_jets = new_corrected_jets

triplet_indices = ak.argcombinations(selected_jets, 3, axis=1, fields=["i", "j", "k"])
triplet_candidates = ak.combinations(selected_jets, 3, axis=1, fields=["a_jet", "b_jet", "c_jet"])

def disjoint_pairs(triplet_indices,triplet_candidates):

    pairs_condidate = ak.combinations(triplet_candidates, 2, axis=1, fields=["a", "b"])
    pairs_indices = ak.combinations(triplet_indices, 2, axis=1, fields=["a_indices", "b_indices"])

    ia, ib = pairs_indices.a_indices, pairs_indices.b_indices

    # Check if the two triplets share any jet indices
    disjoint = ~(
        (ia.i == ib.i) | (ia.i == ib.j) | (ia.i == ib.k) |
        (ia.j == ib.i) | (ia.j == ib.j) | (ia.j == ib.k) |
        (ia.k == ib.i) | (ia.k == ib.j) | (ia.k == ib.k)
    )
    pairs_condidate = pairs_condidate[disjoint]
    pairs_indices = pairs_indices[disjoint]
    
    top_mass_a = (pairs_condidate.a.a_jet + pairs_condidate.a.b_jet + pairs_condidate.a.c_jet).mass
    top_mass_b = (pairs_condidate.b.a_jet + pairs_condidate.b.b_jet + pairs_condidate.b.c_jet).mass

    delta_mass = top_mass_a - top_mass_b
    
    return {
        "pairs_condidate" : pairs_condidate,
        "pairs_indices" : pairs_indices,
        "top_mass_a": top_mass_a,
        "top_mass_b": top_mass_b,
        "delta_mass": delta_mass,
    }

Info_of_disjoint_pairs = disjoint_pairs(triplet_indices,triplet_candidates)

pairs_condidate = Info_of_disjoint_pairs["pairs_condidate"]
pairs_indices = Info_of_disjoint_pairs["pairs_indices"]
top_mass_a = Info_of_disjoint_pairs["top_mass_a"]
top_mass_b = Info_of_disjoint_pairs["top_mass_b"]
delta_mass = Info_of_disjoint_pairs["delta_mass"]

W_jets_a = ak.concatenate([
    pairs_condidate.a["a_jet"][:, :, None],
    pairs_condidate.a["b_jet"][:, :, None],
    pairs_condidate.a["c_jet"][:, :, None]
], axis=2)

W_pairs_a = ak.combinations(W_jets_a, 2, axis=2, fields=["w1_a", "w2_a"])
W_mass_a_candidates = (W_pairs_a["w1_a"] + W_pairs_a["w2_a"]).mass  # shape: [events, 10 tops, 3 W]

W_jets_b = ak.concatenate([
    pairs_condidate.b["a_jet"][:, :, None],
    pairs_condidate.b["b_jet"][:, :, None],
    pairs_condidate.b["c_jet"][:, :, None]
], axis=2)


W_pairs_b = ak.combinations(W_jets_b, 2, axis=2, fields=["w1_b", "w2_b"])
W_mass_b_candidates = (W_pairs_b["w1_b"] + W_pairs_b["w2_b"]).mass  # shape: [events, 10 tops, 3 W]


what_we_need = ak.zip({
        "delta_Mtop": delta_mass,
        "indices_a" : pairs_indices.a_indices,
#         "a_jet_a" : pairs_condidate.a["a_jet"],
#         "b_jet_a" : pairs_condidate.a["b_jet"],
#         "c_jet_a" : pairs_condidate.a["c_jet"],
#         "pairs_indices" : pairs_indices,
        "top_mass_a": top_mass_a,
        "W_mass_a_1" : W_mass_a_candidates[:, :, 0],
        "W_mass_a_2" : W_mass_a_candidates[:, :, 1],
        "W_mass_a_3" : W_mass_a_candidates[:, :, 2],
#         "a_jet_b" : pairs_condidate.b["a_jet"],
#         "b_jet_b" : pairs_condidate.b["b_jet"],
#         "c_jet_b" : pairs_condidate.b["c_jet"],
       "indices_b" : pairs_indices.b_indices,
        "top_mass_b": top_mass_b,
        "W_mass_b_1" : W_mass_b_candidates[:, :, 0],
        "W_mass_b_2" : W_mass_b_candidates[:, :, 1],
        "W_mass_b_3" : W_mass_b_candidates[:, :, 2],
    })
ak.to_dataframe(what_we_need[0])

In [ ]:
import ctypes

def get_event_bin_contents(hist3D, ev_dt, ev_tops, ev_ws):
    results = []

    xaxis = hist3D.GetXaxis()
    yaxis = hist3D.GetYaxis()
    zaxis = hist3D.GetZaxis()

    x_min, x_max = xaxis.GetXmin(), xaxis.GetXmax()
    y_min, y_max = yaxis.GetXmin(), yaxis.GetXmax()
    z_min, z_max = zaxis.GetXmin(), zaxis.GetXmax()

    for dtt, top, ws_list in zip(ev_dt, ev_tops, ev_ws):        
        for w in ws_list:
            if (x_min <= dtt < x_max and
                y_min <= top < y_max and
                z_min <= w < z_max):


                bin_idx = hist3D.FindBin(dtt, top, w)

               
                ix = ctypes.c_int()
                iy = ctypes.c_int()
                iz = ctypes.c_int()

                hist3D.GetBinXYZ(bin_idx, ix, iy, iz)

                bin_x = ix.value
                bin_y = iy.value
                bin_z = iz.value

                x_low = xaxis.GetBinLowEdge(bin_x)
                x_up  = xaxis.GetBinUpEdge(bin_x)
                dx = x_up - x_low

                y_low = yaxis.GetBinLowEdge(bin_y)
                y_up  = yaxis.GetBinUpEdge(bin_y)
                dy = y_up - y_low

                z_low = zaxis.GetBinLowEdge(bin_z)
                z_up  = zaxis.GetBinUpEdge(bin_z)
                dz = z_up - z_low

                bin_volume = dx * dy * dz

                val = hist3D.GetBinContent(bin_idx)
                normalized_val = val * bin_volume 

                results.append({"prob": normalized_val, "inside": True})

            else:

                results.append({"prob": 0.0, "inside": False})

    return results


probs_a_group = ak.Array([
    get_event_bin_contents(hist3D, dt, tops, ws)
    for dt, tops, ws in zip(delta_mass, top_mass_a, W_mass_a_candidates)
])

probs_b_group = ak.Array([
    get_event_bin_contents(hist3D, dt, tops, ws)
    for dt, tops, ws in zip(delta_mass, top_mass_b, W_mass_b_candidates)
])

Info_per_event = ak.zip({
        "delt_Mtop" : ak.flatten(ak.broadcast_arrays(W_mass_a_candidates,delta_mass)[1], axis=2),
        "a_indices" : ak.flatten(ak.broadcast_arrays(W_mass_a_candidates,pairs_indices.a_indices)[1], axis=2) ,
        "a_jet_candidate" : ak.flatten(ak.broadcast_arrays(W_mass_a_candidates,pairs_condidate.a)[1], axis=2) ,
        "a_top_mass": ak.flatten(ak.broadcast_arrays(W_mass_a_candidates,top_mass_a)[1], axis=2),
        "a_W_mass" : ak.flatten(W_mass_a_candidates, axis=2) ,
        "a_prob" : probs_a_group["prob"],
        "a_inside" : probs_a_group["inside"],
        "b_indices" : ak.flatten(ak.broadcast_arrays(W_mass_b_candidates,pairs_indices.b_indices)[1], axis=2) ,
        "b_jet_candidate" : ak.flatten(ak.broadcast_arrays(W_mass_b_candidates,pairs_condidate.b)[1], axis=2) ,
        "b_top_mass": ak.flatten(ak.broadcast_arrays(W_mass_b_candidates,top_mass_b)[1], axis=2),
        "b_W_mass" : ak.flatten(W_mass_b_candidates, axis=2) ,
        "b_prob" : probs_b_group["prob"],
        "b_inside" : probs_b_group["inside"]

})

In [ ]:
def select_best_disjoint_pairs(info_array):

    both_inside = info_array.a_inside & info_array.b_inside
    nonzero_prob = (info_array.a_prob > 0) & (info_array.b_prob > 0)

    # Valid pairs satisfy all conditions
    valid_mask = both_inside & nonzero_prob
    valid_pairs = info_array[valid_mask]

    # Compute joint likelihood
    joint_likelihood = valid_pairs.a_prob * valid_pairs.b_prob
    # Get index of pair with max likelihood per event
    max_indices = ak.argmax(joint_likelihood, axis=1)

    # Select only the best pair per event
    best_pairs = valid_pairs[ak.local_index(valid_pairs) == max_indices]

    # Add joint_likelihood field for reference
    best_pairs = ak.with_field(best_pairs, joint_likelihood[ak.local_index(valid_pairs) == max_indices], where="joint_likelihood")

    return best_pairs

    
best_pairs_clean = select_best_disjoint_pairs(Info_per_event)

top_a = ak.to_numpy(ak.flatten(best_pairs_clean.a_top_mass))
top_b = ak.to_numpy(ak.flatten(best_pairs_clean.b_top_mass))
all_tops = list(top_a) + list(top_b)

W_a = ak.to_numpy(ak.flatten(best_pairs_clean.a_W_mass))
W_b = ak.to_numpy(ak.flatten(best_pairs_clean.b_W_mass))
all_W = list(W_a) + list(W_b)

ttbar_a = ak.to_numpy(ak.flatten(best_pairs_clean.delt_Mtop))
all_delta_top = list(ttbar_a)


In [ ]:
# Convert awkward arrays to numpy arrays 
top_a = np.array(ak.flatten(best_pairs_clean.a_top_mass))
top_b = np.array(ak.flatten(best_pairs_clean.b_top_mass))
all_tops = np.concatenate([top_a, top_b])
final_M_ttbar = best_pairs_clean.a_top_mass + best_pairs_clean.b_top_mass

W_a = np.array(ak.flatten(best_pairs_clean.a_W_mass))
W_b = np.array(ak.flatten(best_pairs_clean.b_W_mass))
all_W = np.concatenate([W_a, W_b])

ttbar_a = np.array(ak.flatten(best_pairs_clean.delt_Mtop))
all_delta_top = ttbar_a  

# -------------------------
# ROOT file for Top masses
f_top = TFile("best_pairs_top.root", "RECREATE")
hist_top_best = TH1F("hist_top_best", "Best Top and AntiTop Masses;Mass [GeV];Events", 50, 50, 220)
for mass in all_tops:
    hist_top_best.Fill(mass)
hist_top_best.Write()
f_top.Close()

# -------------------------
# ROOT file for W masses
f_W = TFile("best_pairs_W.root", "RECREATE")
hist_W_best = TH1F("hist_W_best", "Best W and AntiW Masses;Mass [GeV];Events", 80, 40, 120)
for mass in all_W:
    hist_W_best.Fill(mass)
hist_W_best.Write()
f_W.Close()

# -------------------------
# ROOT file for Top-AntiTop mass difference
f_delta = TFile("best_pairs_deltaM.root", "RECREATE")
hist_delta_best = TH1F("hist_delta_best", "Best Top-AntiTop Mass Difference;DeltaMass [GeV];Events", 50, -100, 100)
for dmass in all_delta_top:
    hist_delta_best.Fill(dmass)
hist_delta_best.Write()
f_delta.Close()


# Compute final M_ttbar (flatten to 1D numpy array)
final_ttbar = np.array(ak.flatten(best_pairs_clean.a_top_mass + best_pairs_clean.b_top_mass))

# Create ROOT file
f_final_ttbar = TFile("best_pairs_final_ttbar.root", "RECREATE")

# Create histogram
hist_final_ttbar = TH1F("hist_final_ttbar", "Final ttbar Mass from Best Pairs;M_ttbar [GeV];Events", 50, 220, 380)

# Fill histogram
for m in final_ttbar:
    hist_final_ttbar.Fill(m)

# Write and close file
hist_final_ttbar.Write()
f_final_ttbar.Close()

In [ ]:
# b = ak.to_dataframe(all_tops)
# b.to_csv("m_for_test.csv", index=False)

df = pd.read_csv("m_for_test.csv")
m = df["values"].values

# ============================
# Histogram
# ============================
counts, edges = np.histogram(m, bins=300)
centers = 0.5 * (edges[:-1] + edges[1:])
errors = np.sqrt(counts)
errors[errors == 0] = 1

# ============================
# Gaussian function
# ============================
def gaussian(x, A, mu, sigma):
    return A * np.exp(-0.5 * ((x - mu) / sigma)**2)

# ============================
# Fit range (x-range)
# ============================
xmin, xmax = 160, 176
mask = (centers >= xmin) & (centers <= xmax)

x_fit   = centers[mask]
y_fit   = counts[mask]
err_fit = errors[mask]

# ============================
# Initial guess
# ============================
p0 = [
    y_fit.max(),                  # A
    170,
    # x_fit[np.argmax(y_fit)],      # mu
    8.0                           # sigma
]

# ============================
# Parameter bounds (YOU control these)
# ============================
A_min, A_max       = 0.5 * y_fit.max(), 100 * y_fit.max()
mu_min, mu_max     = 170.0, 175.0
sigma_min, sigma_max = 0.5, 15.0

bounds = (
    [A_min, mu_min, sigma_min],   # lower bounds
    [A_max, mu_max, sigma_max]    # upper bounds
)

# ============================
# Fit
# ============================
popt, pcov = curve_fit(
    gaussian,
    x_fit,
    y_fit,
    p0=p0,
    sigma=err_fit,
    absolute_sigma=True,
    bounds=bounds,
    maxfev=20000
)

A, mu, sigma = popt
dA, dmu, dsigma = np.sqrt(np.diag(pcov))

# ============================
# Chi2 / ndf
# ============================
fit_vals = gaussian(x_fit, *popt)
chi2 = np.sum(((y_fit - fit_vals) / err_fit)**2)
ndf = len(x_fit) - len(popt)
chi2_ndf = chi2 / ndf

# ============================
# Print results
# ============================
print("====== Gaussian Fit ======")
print(f"A      = {A:.3f} ± {dA:.3f}")
print(f"mu     = {mu:.3f} ± {dmu:.3f}")
print(f"sigma  = {sigma:.3f} ± {dsigma:.3f}")
print(f"chi2/ndf = {chi2_ndf:.3f}")

# ============================
# Plot
# ============================
plt.errorbar(centers, counts, yerr=errors, fmt='o', markersize=3, label="Data")
plt.plot(x_fit, fit_vals, 'g-', lw=2, label="Gaussian fit")

plt.xlabel("m")
plt.ylabel("Counts")
plt.legend()
plt.show()
